# Lesson 03 - Agentic Design Patterns

In dieser Lektion untersuchen wir drei grundlegende Designmuster für den Bau effektiver KI-Agenten:

1. **Klare Agentenanweisungen** — Erstellung präziser, rollen-definierender Eingabeaufforderungen, die das Verhalten des Agenten steuern  
2. **Strukturierte Ausgabe mit Pydantic-Modellen** — Sicherstellung, dass Agenten vorhersehbare, validierte Daten zurückgeben  
3. **Agenten mit Einzelverantwortung** — Design fokussierter Agenten, die jeweils eine Aufgabe gut erfüllen  

Wir wenden jedes Muster auf ein **Reiseziel-Empfehlungsszenario** an und bauen schrittweise ein System auf, das Ziele vorschlagen, Verfügbarkeiten prüfen und Logistik handhaben kann.


## Einrichtung


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Muster 1: Klare Anweisungen für den Agenten

Das wirkungsvollste Muster ist auch das einfachste: klare, detaillierte Anweisungen für Ihren Agenten zu schreiben.

Gute Anweisungen definieren:
- **Wer** der Agent ist (Persona und Tonfall)
- **Was** er tun soll (Schritt-für-Schritt-Verantwortlichkeiten)
- **Wie** er sich verhalten soll (Einschränkungen und Stil)

Im Folgenden erstellen wir einen Reise-Concierge-Agenten mit expliziten Anweisungen, die jede Antwort, die er erzeugt, gestalten.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Muster 2: Strukturierte Ausgabe mit Pydantic-Modellen

Freier Text ist für Unterhaltungen nützlich, aber nachgelagerte Systeme benötigen strukturierte Daten.
Indem wir **Pydantic-Modelle** mit einer **Werkzeugfunktion** kombinieren, können wir:

- Ein genaues Schema für die Ausgabe des Agenten definieren
- Antworten automatisch validieren
- Agentenergebnisse zuverlässig in die Anwendungslogik integrieren

Wir stellen auch ein Werkzeug vor, das Reisezieldetails zurückgibt, damit der Agent seine Empfehlungen auf reale Daten stützt.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Muster 3: Single-Responsibility-Agenten

Komplexe Aufgaben profitieren davon, die Arbeit auf mehrere fokussierte Agenten mit jeweils einer einzigen Verantwortung aufzuteilen:

- Ein **Zielort-Experte**, der sich mit Orten und Verfügbarkeiten auskennt
- Ein **Logistikplaner**, der Flüge, Hotels und Reiserouten organisiert

Dies spiegelt das Software-Engineering-Prinzip der *Trennung der Verantwortlichkeiten* wider — jeder Agent ist leichter unabhängig zu testen, zu warten und zu verbessern.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Zusammenfassung

In dieser Lektion haben wir drei agentenbasierte Designmuster auf ein Reiseempfehlungsszenario angewendet:

| Muster | Kernidee | Vorteil |
|---|---|---|
| **Klare Anweisungen** | Persona, Verantwortlichkeiten und Einschränkungen von Anfang an definieren | Konsistentes, markenkonformes Agentenverhalten |
| **Strukturierte Ausgabe** | Pydantic-Modelle als Antwortformat verwenden | Validierte, maschinenlesbare Ergebnisse |
| **Single Responsibility** | Jedem Agenten eine fokussierte Aufgabe geben | Einfacheres Testen, Warten und Kombinieren |

Diese Muster lassen sich natürlich miteinander kombinieren — Sie können klare Anweisungen mit strukturierter Ausgabe in einem Single-Responsibility-Agenten verbinden, um robuste, produktionsfertige Systeme zu erstellen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:  
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in der Ausgangssprache gilt als maßgebliche Quelle. Für wichtige Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die durch die Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
